# 🌧️ BharatBench — Baseline Models: Persistence, Climatology & Weekly Climatology

**Notebook:** `1_climatology_persistence.ipynb`  
**Dataset:** BharatBench (IMDAA reanalysis, 1990–2020, 1.08° resolution, 32×32 grid)  
**Paper:** *Preparing benchmarks for data-driven weather forecasting system over India*  
**Dataset DOI / Download:** [Kaggle — maslab/bharatbench](https://www.kaggle.com/datasets/maslab/bharatbench)  
**Code Repository:** [GitHub — MASLABnitrkl/BharatBench](https://github.com/MASLABnitrkl/BharatBench)

---

## Purpose

This notebook computes three simple, non-learned **baseline forecasts** that every data-driven model must outperform to be considered skillful:

| Baseline | Description |
|---|---|
| **Persistence** | Assume the atmosphere does not change; forecast = current state |
| **Climatology** | Use the long-term daily mean (1990–2017) as the forecast |
| **Weekly Climatology** | Use the long-term weekly mean (1990–2017) as the forecast |

All baselines are evaluated on the **test period (2019–2020)** using three metrics: RMSE, MAE, and ACC.

---

## Variables evaluated

| Short name | Variable | Level |
|---|---|---|
| `HGT_prl` | Geopotential Height | 500 hPa |
| `TMP_prl` | Temperature | 850 hPa |
| `TMP_2m` | Temperature | 2 m surface |
| `APCP_sfc` | 6-hourly Accumulated Precipitation | Surface |

---

## Notebook structure

1. [Setup & Imports](#1-setup)  
2. [Load Dataset](#2-load-dataset)  
3. [Spatial Overview Plot](#3-spatial-overview)  
4. [Evaluation Metrics](#4-metrics)  
5. [Dataset Splits](#5-splits)  
6. [Baseline 1 — Persistence](#6-persistence)  
7. [Baseline 2 — Climatology](#7-climatology)  
8. [Baseline 3 — Weekly Climatology](#8-weekly-climatology)

---
<a id="1-setup"></a>
## 1. Setup & Imports

All required libraries are imported below. The notebook uses standard scientific Python packages.  
`warnings.filterwarnings('ignore')` suppresses benign xarray/numpy deprecation warnings.

In [ ]:
# Libraries 
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

> **Environment check:** If any import fails, install the missing package with:
> ```bash
> pip install numpy xarray matplotlib pandas cartopy
> ```

---
<a id="2-load-dataset"></a>
## 2. Load Dataset

The BharatBench dataset is stored as a single **NetCDF** file containing all surface and pressure-level variables for 1990–2020.

> ⚠️ **Update the file path below** to match the location of your downloaded dataset.  
> The dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench).

**Dataset specifications:**
- Spatial domain: 5°N – 40°N, 65°E – 100°E  
- Spatial resolution: 1.08° (~32 × 32 grid)  
- Temporal resolution: 6-hourly (00, 06, 12, 18 UTC)  
- Period: 1990–2020

In [ ]:
ds=  xr.open_dataset(r"G:/IMDAA_Regrid_1.08_1990_2022/IMDAA_merged_1.08_1990_2020.nc")
ds

---
<a id="3-spatial-overview"></a>
## 3. Spatial Overview Plot

The cell below plots the **temporal mean of 6-hourly accumulated precipitation (TP6h)** over the study domain using Cartopy. This provides a quick visual sanity-check of the dataset's spatial coverage and data values.

> 💡 The plot is saved as `TP6h.png` in the working directory at 600 DPI.

In [ ]:
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [ ]:
import cartopy.feature as cfeature
import cartopy.crs as ccrs

In [ ]:

mean_all = ds.mean(dim='time')
z500 = mean_all
data_values = z500['APCP_sfc'].values

# Get the extent of your xarray dataset
lon_min, lon_max, lat_min, lat_max = z500.lon.min(), z500.lon.max(), z500.lat.min(), z500.lat.max()

# Create a basic plot within the extent of your xarray data using cartopy
fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
im = ax.imshow(data_values, cmap='jet', origin='lower', extent=[lon_min, lon_max, lat_min, lat_max], transform=ccrs.PlateCarree())

ax.tick_params( labelcolor='black', labelsize='large', width=2)
# Add country borders and coastlines within the extent
ax.add_feature(cfeature.COASTLINE, linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=1)
#world.boundary.plot(ax=ax, linewidth=1, color='black')

# Add Cartopy graticules (latitude and longitude lines)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlabels_top = False
gl.ylabels_right = False
gl.xformatter = LONGITUDE_FORMATTER
gl.yformatter = LATITUDE_FORMATTER
gl.ylabel_style = {'color': 'black', 'weight': 'bold'}
gl.xlabel_style = {'color': 'black', 'weight': 'bold'}
#ax.set_xticks(range(int(lon_min), int(lon_max) + 1, 10), crs=ccrs.PlateCarree())
#ax.set_yticks(range(int(lat_min), int(lat_max) + 1, 10), crs=ccrs.PlateCarree())
# plt.xticks(fontweight='bold')
# plt.yticks(fontweight='bold')
cbar = plt.colorbar(im)
cbar.ax.set_ylabel('(Kg/m^2)', fontsize = 12, weight="bold")
cbar.ax.set_yticklabels(cbar.ax.get_yticklabels(), weight="bold")
plt.title('Mean TP6h ' , fontweight='bold')
# plt.xlabel('Longitude')
# plt.ylabel('Latitude')
# plt.grid(True)
plt.savefig('TP6h.png', dpi = 600)
plt.show()

---
<a id="4-metrics"></a>
## 4. Evaluation Metrics

Three standard verification metrics are defined and used throughout this notebook.

### 4.1 Root Mean Square Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{N_{\text{pred}}} \sum_{i}\frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} (f_{i,j,k} - t_{i,j,k})^2}$$

RMSE penalises large errors more heavily than MAE. It is the **primary ranking metric** for BharatBench.

In [ ]:
def compute_rmse(prediction, actual,  mean_dims = ('time', 'latitude', 'longitude')):
  error = prediction - actual
  rmse = np.sqrt(((error)**2 ).mean(mean_dims))
  return rmse

### 4.2 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{N_{\text{pred}}} \sum_{i}\frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} |f_{i,j,k} - t_{i,j,k}|$$

MAE treats all errors equally and is more robust to outliers than RMSE.

In [ ]:
def compute_mae(prediction, actual, mean_dims = ('time', 'latitude', 'longitude')):
    error = prediction - actual
    mae = np.abs(error).mean(mean_dims)
    return mae

### 4.3 Anomaly Correlation Coefficient (ACC)

$$\text{ACC} = \frac{\sum_{i,j,k} f'_{i,j,k}\, t'_{i,j,k}}{\sqrt{\sum_{i,j,k} f'^2_{i,j,k} \cdot \sum_{i,j,k} t'^2_{i,j,k}}}$$

where primed variables denote departures from climatology.

| ACC range | Interpretation |
|---|---|
| > 0.8 | Highly skillful forecast |
| ≈ 0.6 | Useful forecast |
| ≈ 0.5 | Comparable to climatological mean |
| < 0.3 | Poor skill; limited forecast value |

> **Note:** An ACC of 0.6 is conventionally considered the threshold of a useful numerical weather forecast.

In [ ]:
def compute_acc(prediction, actual):
    clim = actual.mean('time')
    try:
        t = np.intersect1d(prediction.time, actual.time)
        pred_anomaly = prediction.sel(time=t) - clim
    except AttributeError:
        t = actual.time.values
        pred_anomaly = prediction - clim
    act_anomaly = actual.sel(time=t) - clim
    
    pred_norm = pred_anomaly - pred_anomaly.mean()
    act_norm = act_anomaly - act_anomaly.mean()

    acc = (
            np.sum(pred_norm * act_norm) /
            np.sqrt(
                np.sum(pred_norm ** 2) * np.sum(act_norm ** 2)
            )
    )
    return acc

---
<a id="5-splits"></a>
## 5. Dataset Splits

The dataset is divided into three non-overlapping periods. A continuous temporal split is used — rather than a random split — because meteorological variables have strong temporal autocorrelation.

| Split | Period | Purpose |
|---|---|---|
| **Training** | 1990–2017 (28 years) | Compute climatologies; train ML models |
| **Validation** | 2018 (1 year) | Hyper-parameter tuning during model training |
| **Test** | 2019–2020 (2 years) | Final evaluation of all baselines and models |

> ⚠️ The test set is held out and must **not** influence any design decision.

In [ ]:
# training dataset selection
train_years = slice('1990', '2017')
# validation dataset selection (this dataset helps with overfitting)
valid_years = slice('2018', '2018')
# test dataset selection
test_years = slice('2019', '2020')

The four target variables are listed below. Note that `HGT_prl` and `TMP_prl` refer to pressure-level fields; the specific level (500 hPa and 850 hPa respectively) is selected when the dataset is loaded.

In [ ]:
var_name = ['HGT_prl', 'TMP_prl', 'TMP_2m', 'APCP_sfc'] # [H500, T850, T2m, TP6h]

---
<a id="6-persistence"></a>
## 6. Baseline 1 — Persistence Forecast

### What is persistence?

The persistence forecast assumes **the atmosphere does not change**: the forecast for time $t + \Delta t$ is simply the observation at time $t$.

$$\hat{x}(t + \Delta t) = x(t)$$

Despite its simplicity, persistence is a competitive baseline at short lead times (< 1–2 days) for slowly-varying fields like Z500. For precipitation, skill degrades rapidly.

### Lead times

The dataset has 4 observations per day (at 00, 06, 12, 18 UTC). Lead times are evaluated from **1 day (4 time steps)** to **15 days (60 time steps)** at daily intervals.

In [ ]:
# Each day the data has four observations at 00 UTC, 06 UTC, 12 UTC and 18 UTC
lead_time_steps = np.arange(4, 64, 4) 
lead_time_steps

### Compute persistence errors

For each variable and each lead time, we:
1. Take the test-set observations at time $t_0$ as the forecast.
2. Compare them against the actual observations at $t_0 + \Delta t$.
3. Record RMSE, MAE, and ACC.

This produces a table with one row per lead time and columns for each variable × metric combination.

In [ ]:
# Compute the rmse for each lead_time_steps

df_error = pd.DataFrame()
for var in var_name:
    error_rmse = []
    error_mae = []
    error_acc = []
    for i,j in enumerate(lead_time_steps):

 # compute persistent forecast
        persistence_fc = ds.sel(time=test_years).isel(time=slice(0, -j))
        persistence_fc['time'] = persistence_fc.time + np.timedelta64(i+1, 'D').astype('timedelta64[ns]')
        target = ds.sel(time=test_years)
        target = target.isel(time=slice(j, None))
        # change the variable name according to the dataset
        error = compute_rmse(persistence_fc, target)[var].values.item() 
        error_rmse.append(error)
        error = compute_mae(persistence_fc, target)[var].values.item()
        error_mae.append(error)
        error = compute_acc(persistence_fc, target)[var].values.item()
        error_acc.append(error) 
    df_error[var+'_RMSE'] = error_rmse
    df_error[var+'_MAE'] = error_mae
    df_error[var+'_ACC'] = error_acc
    

### Inspect the full error table

Each row corresponds to one lead time step (6 hours apart). Column names follow the pattern `<variable>_<metric>`.

In [ ]:
df_error

### Extract 3-day and 5-day lead time rows

Rows at index 2 and 4 correspond to lead times of **3 days** and **5 days** — the two benchmark lead times used throughout the paper.

In [ ]:
df_error.iloc[[2, 4]]

### Prepare plotting DataFrame

Build a tidy DataFrame for the precipitation variable (`APCP_sfc`) for visualisation. Lead days run from 1 to 15.

In [ ]:
df = pd.DataFrame({ 'Lead Days': np.arange(1, 16, 1),'RMSE': df_error['APCP_sfc_RMSE'], 'MAE': df_error['APCP_sfc_MAE'], 'ACC': df_error['APCP_sfc_ACC']})
df.head()


### Plot persistence skill vs lead time

The dual-axis plot below shows:
- **Left axis (blue / green):** RMSE and MAE — both should increase with lead time as forecast skill degrades.
- **Right axis (red):** ACC — should decrease with lead time.

> 💾 The figure is saved to the path specified in `plt.savefig(...)`. Update this path as needed.

In [ ]:
fig, ax1 = plt.subplots()

plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold' )

ax1.plot(df['Lead Days'], df['RMSE'], label='RMSE', color='blue')
ax1.plot(df['Lead Days'], df['MAE'], label='MAE', color='green')
ax1.set_xlabel('Lead Days', fontweight='bold')
ax1.set_ylabel('ERROR (m)', color='black', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='black', labelsize='large', width=2 )
ax1.tick_params(axis='x', labelcolor='black', labelsize='large', width=2)

ax2 = ax1.twinx()
ax2.plot(df['Lead Days'], df['ACC'], label='ACC', color='red')
ax2.set_ylabel('ACC', color='red', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='red', labelsize='large', width=2)


lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
lines = lines1 + lines2
labels = labels1 + labels2

legend = ax1.legend(lines, labels, loc='upper right', bbox_to_anchor=(0.99, 0.88))
# Make legend labels bold
for label in legend.get_texts():
    label.set_fontweight('bold')

 
ax1.set_title('6 hourly accumulate precipitation', fontweight='bold')

plt.yticks(fontweight='bold' )

plt.savefig(r'D:\VSCODE_Works\BharatBench\ignore\Figures\APCP_sfc_persistence.png', dpi=300)

plt.show()

---
<a id="7-climatology"></a>
## 7. Baseline 2 — Climatology

### What is climatology?

The climatological forecast uses the **long-term daily mean** computed over the training period (1990–2017) as the forecast for every year.

$$\hat{x}(\text{day } d) = \frac{1}{N_{\text{train}}} \sum_{y \in \text{train}} x(y, d)$$

This captures the **seasonal cycle** but ignores year-to-year variability. It is a useful baseline because it represents the best forecast achievable with zero dynamical information.

### Implementation

`groupby('time.dayofyear').mean()` computes 365 daily means (one per calendar day) over the training period. These are then matched to the test-set dates by day-of-year.

In [ ]:
clim = ds.sel(time=train_years).groupby('time.dayofyear').mean()
error_rmse = compute_rmse(clim.sel(dayofyear=ds.sel(time=test_years).time.dt.dayofyear), ds.sel(time=test_years))
error_mae = compute_mae(clim.sel(dayofyear=ds.sel(time=test_years).time.dt.dayofyear), ds.sel(time=test_years))
error_acc = compute_acc(clim.sel(dayofyear=ds.sel(time=test_years).time.dt.dayofyear), ds.sel(time=test_years)) 

### Error summary table

Rows: RMSE, MAE, ACC. Columns: one per variable. All metrics are computed over the full test period (2019–2020).

In [ ]:
df_error = pd.DataFrame()
for var in var_name:
    df_error[var] = pd.DataFrame({var: [error_rmse[var].values, error_mae[var].values, error_acc[var].values]}, index=['RMSE', 'MAE', 'ACC'])
df_error    

### Print formatted results

A human-readable summary for each variable is printed below.

In [ ]:
print(f" Geopotential  Height at 500hPa \n RMSE : {error_rmse['HGT_prl'].values}\n MAE : {error_mae['HGT_prl'].values}\n ACC : {error_acc['HGT_prl'].values} ")
print(f" Temperature at  850hPa \n RMSE : {error_rmse['TMP_prl'].values}\n MAE : {error_mae['TMP_prl'].values}\n ACC : {error_acc['TMP_prl'].values} ")
print(f" Total Precipitation \n RMSE : {error_rmse['APCP_sfc'].values}\n MAE : {error_mae['APCP_sfc'].values}\n ACC : {error_acc['APCP_sfc'].values} ")
print(f" 2m Temperature \n RMSE : {error_rmse['TMP_2m'].values}\n MAE : {error_mae['TMP_2m'].values}\n ACC : {error_acc['TMP_2m'].values} ")

---
<a id="8-weekly-climatology"></a>
## 8. Baseline 3 — Weekly Climatology

### What is weekly climatology?

Weekly climatology averages observations by **ISO week number** (weeks 1–52) rather than calendar day. This provides a slightly smoother seasonal cycle, reducing the noise associated with day-of-year grouping for short training records.

$$\hat{x}(\text{week } w) = \frac{1}{N_{\text{train}}} \sum_{y \in \text{train}} x(y, w)$$

In practice, weekly and daily climatologies produce very similar skill scores — both serve as lower bounds that data-driven models are expected to exceed.

### Implementation

`compute_weekly_climatology` maps each test-set timestamp to its ISO week and looks up the corresponding weekly mean from the training set.

In [ ]:
# computation of weekly climatology
def compute_weekly_climatology(ds_train, valid_time):
    ds_train['week'] = ds_train['time.week']
    weekly_averages = ds_train.groupby('week').mean('time')
    valid_time['week'] = valid_time['time.week']
    fc_list = []
    for t in valid_time:
        fc_list.append(weekly_averages.sel(week=t.week))
    return xr.concat(fc_list, dim=valid_time)

Prepare training and test data slices for the weekly climatology computation.

In [ ]:
train_data = ds.sel(time=train_years)
target = ds.sel(time=test_years)

Compute weekly climatology forecasts for the entire test period. This may take a moment as it iterates over all test time steps.

In [ ]:
weekly_climatology = compute_weekly_climatology(train_data, target.time)

Compute RMSE, MAE, and ACC against the test-set observations.

In [ ]:
error_rmse = compute_rmse(weekly_climatology, target)
error_mae = compute_mae(weekly_climatology, target)
error_acc = compute_acc(weekly_climatology, target)

### Error summary table

In [ ]:
df_error = pd.DataFrame()
for var in var_name:
    df_error[var] = pd.DataFrame({var: [error_rmse[var].values, error_mae[var].values, error_acc[var].values]}, index=['RMSE', 'MAE', 'ACC'])
df_error 

### Print formatted results

In [ ]:
print(f" Geopotential  Height at 500hPa \n RMSE : {error_rmse['HGT_prl'].values}\n MAE : {error_mae['HGT_prl'].values}\n ACC : {error_acc['HGT_prl'].values} ")
print(f" Temperature at  850hPa \n RMSE : {error_rmse['TMP_prl'].values}\n MAE : {error_mae['TMP_prl'].values}\n ACC : {error_acc['TMP_prl'].values} ")
print(f" Total Precipitation \n RMSE : {error_rmse['APCP_sfc'].values}\n MAE : {error_mae['APCP_sfc'].values}\n ACC : {error_acc['APCP_sfc'].values} ")
print(f" 2m Temperature \n RMSE : {error_rmse['TMP_2m'].values}\n MAE : {error_mae['TMP_2m'].values}\n ACC : {error_acc['TMP_2m'].values} ")

---
## Summary & Next Steps

This notebook has established the three non-learned baseline scores for BharatBench.

| Baseline | Z500 RMSE | T850 RMSE | T2m RMSE | TP6h RMSE |
|---|---|---|---|---|
| Persistence (3-day) | — | — | — | — |
| Climatology | — | — | — | — |
| Weekly Climatology | — | — | — | — |

> ✏️ Fill in the table above with your computed values once the notebook has run.

### Interpreting results

- Persistence errors grow with lead time; climatology errors are lead-time independent.
- Weekly climatology is virtually identical to climatology — this confirms that the seasonal cycle is well-captured by both grouping strategies.
- **Any data-driven model (CNN, ConvLSTM, etc.) must achieve RMSE and ACC values better than all three baselines to demonstrate skill.**

### Continuing with BharatBench

| Notebook | Description |
|---|---|
| `2_linear_regression.ipynb` | Linear regression baseline |
| `3_CNN_ConvLSTM.ipynb` | CNN, ConvLSTM encoder-decoder model | 


---
*BharatBench — MAS Lab, NIT Rourkela. Dataset: [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench) · Code: [GitHub](https://github.com/MASLABnitrkl/BharatBench)*